In [ ]:

import os
import zipfile

ZIP_FILE = "archive.zip"

DATASET_FOLDER = "dataset"

os.makedirs(DATASET_FOLDER, exist_ok=True)

with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
    zip_ref.extractall(DATASET_FOLDER)

print("zip extract")



In [ ]:

dataset_path = DATASET_FOLDER

print("Dataset Path :", dataset_path)


classes = os.listdir(dataset_path)

print("\nClasses Found :")

for class_name in classes:
    print(class_name)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pprint
import joblib
from skimage.feature import hog
from skimage.feature import local_binary_pattern
import cv2
import os
from sklearn.svm import SVC

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
IMG_SIZE = (128,128)
TEST_SIZE = 0.2

In [ ]:
def extract_features(img_path):
    img = cv2.imread(img_path)
    if img is None:
        return None
    img = cv2.resize(img,IMG_SIZE)
    gray = cv2.cvtColor(img,cv2.COLOR_RGB2GRAY)
    feats = []


    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    for ch in cv2.split(hsv):
        feats += [ch.mean(), ch.std()]


    lbp = local_binary_pattern(gray,P=8, R=1, method = "uniform")
    hist, _ = np.histogram(lbp, bins = 10, range=(0,10), density = True)
    feats += list(hist)

    edges = cv2.Canny(gray,100,200)
    feats.append(edges.mean())

    moments = cv2.moments(gray)
    hu = cv2.HuMoments(moments).flatten()
    feats+= list(-np.sign(hu)*np.log10(np.abs(hu) + 1e-10))

    cnts, _ = cv2.findContours(edges,cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if cnts:
        c = max(cnts,key=cv2.contourArea)
        feats += [cv2.contourArea(c), cv2.arcLength(c,True)]
    else:
        feats += [0,0]

    hog_feats = hog(gray, orientations = 9, pixels_per_cell = (8,8), cells_per_block = (2,2), feature_vector=True)
    feats += list(hog_feats)

    return np.array(feats, dtype=np.float32)
    

In [ ]:
X,y = [],[]
cat_folder = "/kaggle/input/competitions/ai-309-e-icv-mse-2/train/train/Cat"
for file in os.listdir(cat_folder):
    path = os.path.join(cat_folder,file)
    f = extract_features(path)
    if f is not None:
        X.append(f)
        y.append("Cat")
    else:
        print(f"skip corrupted{file}")
dog_folder = "/kaggle/input/competitions/ai-309-e-icv-mse-2/train/train/Dog"
for file in os.listdir(dog_folder):
    path = os.path.join(dog_folder,file)
    f = extract_features(path)
    if f is not None:
        X.append(f)
        y.append("Dog")
    else:
        print(f"skip corrupted{file}")

In [ ]:
X = np.array(X)
y = np.array(y)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = TEST_SIZE, random_state=42, stratify=y)

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
clf = SVC(kernel='linear')
clf.fit(X_train, y_train)

In [ ]:
clf = RandomForestClassifier(n_estimators = 100, random_state=42)
clf.fit(X_train, y_train)

In [ ]:
y_pred = clf.predict(X_test)
classes = ["Cat","Dog"]
print(classification_report(y_test,y_pred, target_names=classes))

In [ ]:
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy :", round(accuracy * 100, 2), "%")